In [ ]:
!pip install transformers torch sentencepiece datasets

In [ ]:
# =====================================================================
# DỰ ÁN: DỊCH TỰ ĐỘNG TIẾNG Ê ĐÊ SANG TIẾNG VIỆT VỚI NLLB-200
# =====================================================================

# Nhập các thư viện cần thiết
import torch                           # Thư viện lõi để tính toán ma trận (trái tim của AI)
import unicodedata                     # Thư viện xử lý các ký tự chữ viết đặc biệt
import datasets                        # Thư viện quản lý dữ liệu (chuẩn bị cho bước huấn luyện sau này)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Công cụ tải mô hình từ Hugging Face

# ---------------------------------------------------------------------
# BƯỚC 1: CẤU HÌNH PHẦN CỨNG (Sử dụng GPU T4 của Google Colab)
# Giải thích cho báo cáo: AI cần tính toán hàng triệu phép toán cùng lúc.
# GPU (Card đồ họa) làm việc này nhanh hơn CPU (Chíp máy tính) hàng chục lần.
# ---------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang sử dụng thiết bị tính toán: {device.upper()}")

# ---------------------------------------------------------------------
# BƯỚC 2: TẢI MÔ HÌNH VÀ BỘ TÁCH TỪ (TOKENIZER)
# Mô hình: facebook/nllb-200-distilled-600M (Bản thu gọn với 600 triệu tham số)
# ---------------------------------------------------------------------
model_name = "facebook/nllb-200-distilled-600M"
print(f"[*] Đang tải Tokenizer và Model từ: {model_name}...")

# Tokenizer giống như "cuốn từ điển", giúp máy tính băm câu văn thành các con số.
# Bắt buộc khai báo src_lang (ngôn ngữ gốc). Vì NLLB-200 chưa hỗ trợ sẵn tiếng Ê Đê,
# ta mượn tạm mã của tiếng Anh hoặc một ngôn ngữ Latinh để mô hình không bị lỗi.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tải mô hình trí tuệ nhân tạo (bộ não) và đưa nó lên GPU để chạy nhanh hơn
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("[*] Tải mô hình thành công!\n")

# ---------------------------------------------------------------------
# BƯỚC 3: HÀM TIỀN XỬ LÝ VĂN BẢN (PRE-PROCESSING)
# ---------------------------------------------------------------------
def chuan_hoa_tieng_ede(text):
    """
    Hàm này dùng để chuẩn hóa bảng mã Unicode.
    Giải thích cho báo cáo: Tiếng Ê Đê có nhiều ký tự đặc biệt (ĕ, ĭ, ŏ, ŭ, ñ, č).
    Trên máy tính, một chữ 'ĕ' có thể được tạo ra bằng 2 cách:
    1. Chữ 'e' + dấu mũ trăng khuyết (Tổ hợp).
    2. Một ký tự 'ĕ' duy nhất (Dựng sẵn).
    Nếu không chuẩn hóa về 1 kiểu (NFC - Dựng sẵn), AI sẽ nghĩ đây là 2 chữ khác nhau
    và học sai lệch. Bước này cực kỳ quan trọng trong NLP.
    """
    text_sach = unicodedata.normalize('NFC', text.strip())
    return text_sach

# ---------------------------------------------------------------------
# BƯỚC 4: HÀM DỊCH THUẬT (INFERENCE)
# ---------------------------------------------------------------------
def dich_ede_sang_viet(cau_ede):
    """
    Hàm nhận vào câu tiếng Ê Đê và ép mô hình dịch ra tiếng Việt.
    """
    # 1. Làm sạch văn bản đầu vào
    cau_da_chuan_hoa = chuan_hoa_tieng_ede(cau_ede)

    # [MỚI THÊM] Gán ngôn ngữ gốc (src_lang).
    # Vì Ê Đê chưa có mã chính thức, ta mượn mã tiếng Indonesia ('ind_Latn')
    # vì nó cùng hệ chữ Latinh và có cấu trúc từ khá gần gũi, giúp Tokenizer cắt từ chuẩn hơn tiếng Anh.
    tokenizer.src_lang = "ind_Latn"

    # 2. Tokenize (Biến chữ thành ma trận số)
    inputs = tokenizer(cau_da_chuan_hoa, return_tensors="pt").to(device)

    # 3. Lấy mã ngôn ngữ đích (Tiếng Việt) - ĐÃ SỬA LỖI
    # Sử dụng convert_tokens_to_ids thay cho lang_code_to_id
    ma_tieng_viet = tokenizer.convert_tokens_to_ids("vie_Latn")

    # 4. Sinh văn bản (Dịch thuật)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=ma_tieng_viet,
            max_length=128
        )

    # 5. Giải mã
    cau_tieng_viet = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return cau_tieng_viet, cau_da_chuan_hoa

# ---------------------------------------------------------------------
# BƯỚC 5: HÀM KIỂM TRA TOKENIZER (CHỨNG MINH CHO BAN GIÁM KHẢO)
# ---------------------------------------------------------------------
def phan_tich_hieu_biet_tu_vung(cau_ede):
    """
    Hàm này mổ xẻ xem AI nhìn thấy câu Ê Đê dưới dạng nào.
    Giải thích cho báo cáo: Mô hình NLLB sử dụng thuật toán BPE (Byte-Pair Encoding).
    Nếu AI chưa từng học tiếng Ê Đê, nó sẽ cắt các từ lạ thành những mẩu chữ cái nhỏ hơn (subwords)
    chứ không để `<unk>` (unknown - không biết). Điều này giúp AI đoán nghĩa tốt hơn.
    """
    cau_da_chuan_hoa = chuan_hoa_tieng_ede(cau_ede)

    # Cắt từ (Tokens)
    tokens = tokenizer.tokenize(cau_da_chuan_hoa)
    # Biến từ thành số (Token IDs)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    print("\n--- PHÂN TÍCH TOKENIZER ---")
    print(f"Câu gốc: {cau_da_chuan_hoa}")
    print(f"Các Tokens (mẩu từ) AI nhìn thấy: {tokens}")
    print(f"Các con số (IDs) tương ứng đưa vào ma trận: {token_ids}")
    print("---------------------------\n")

# =====================================================================
# CHẠY THỬ NGHIỆM THỰC TẾ
# =====================================================================

# Câu mẫu tiếng Ê Đê (Ví dụ: "Klei bhiăn mnia mtao" - Phong tục tập quán / hoặc một câu chào)
# Bạn có thể thay đổi câu này bằng dữ liệu thực tế của bạn
cau_thu_nghiem = "Klei čih pioh hriăm ti hră" # Ví dụ giả định có các chữ đặc biệt như č, ă

# 1. Chạy phân tích Tokenizer để đưa vào báo cáo
phan_tich_hieu_biet_tu_vung(cau_thu_nghiem)

# 2. Chạy dịch thuật
print("--- KẾT QUẢ DỊCH THUẬT ---")
ket_qua_dich, cau_chuan = dich_ede_sang_viet(cau_thu_nghiem)
print(f"[Đầu vào - Ê Đê]: {cau_chuan}")
print(f"[Đầu ra - Tiếng Việt]: {ket_qua_dich}")

[*] Đang sử dụng thiết bị tính toán: CUDA
[*] Đang tải Tokenizer và Model từ: facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[*] Tải mô hình thành công!


--- PHÂN TÍCH TOKENIZER ---
Câu gốc: Klei čih pioh hriăm ti hră
Các Tokens (mẩu từ) AI nhìn thấy: ['▁K', 'lei', '▁č', 'ih', '▁pi', 'oh', '▁hri', 'ăm', '▁ti', '▁hr', 'ă']
Các con số (IDs) tương ứng đưa vào ma trận: [98, 5874, 1136, 1290, 780, 1046, 26232, 4005, 206, 7764, 248427]
---------------------------

--- KẾT QUẢ DỊCH THUẬT ---
[Đầu vào - Ê Đê]: Klei čih pioh hriăm ti hră
[Đầu ra - Tiếng Việt]: Clei čih pioh hriăm ti hră


In [ ]:
import os
import pandas as pd
from google.colab import files

# 1. Mở cửa sổ chọn file CSV từ máy tính
print("[*] Bấm vào nút 'Choose Files' / 'Chọn tệp' bên dưới để nạp file CSV của bạn:")
uploaded = files.upload()

# 2. Lấy tên file vừa tải lên và đổi thành 'du_lieu_song_ngu.csv'
ten_file_goc = list(uploaded.keys())[0]
if ten_file_goc != "du_lieu_song_ngu.csv":
    if os.path.exists("du_lieu_song_ngu.csv"):
        os.remove("du_lieu_song_ngu.csv")
    os.rename(ten_file_goc, "du_lieu_song_ngu.csv")

# 3. Đọc dữ liệu kiểm tra
df = pd.read_csv("du_lieu_song_ngu.csv")
print(f"\n🎉 [THÀNH CÔNG] Đã nạp thành công {len(df)} dòng dữ liệu vào hệ thống!")
print("--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---")
df.head()

[*] Bấm vào nút 'Choose Files' / 'Chọn tệp' bên dưới để nạp file CSV của bạn:


Saving du_lieu_song_ngu.csv to du_lieu_song_ngu.csv

🎉 [THÀNH CÔNG] Đã nạp thành công 408 dòng dữ liệu vào hệ thống!
--- 5 DÒNG DỮ LIỆU ĐẦU TIÊN ---


,id,text_ede,text_vi
0,1,Anei jing gưl nao čhưn klă sĭt tal êlâo mơ̆ng ...,Đây là chuyến công du nước ngoài đầu tiên của ...
1,2,"Ti Lào, klei nao čhưn mkŏ mjing mtam leh ala č...","Tại Lào, chuyến thăm nhằm làm sâu sắc hơn quan..."
2,3,"Ti Campuchia, gưl nao čhưn mkŏ mjing pă mlan l...","Tại Campuchia, chuyến thăm kéo dài ba ngày và ..."
3,4,"Hlăm gưl nao čhưn, Khua knŭk kna Lê Minh Hưng ...","Trong chuyến thăm, Thủ tướng Lê Minh Hưng sẽ h..."
4,5,Gưl nao čhưn đru ba dŭm klei kuôl kă gưl dlông...,Chuyến công tác góp phần củng cố tình hữu nghị...


In [ ]:
!pip install transformers datasets evaluate sacrebleu accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.0 MB/s eta 0:00:00


In [ ]:
!pip install requests beautifulsoup4 pandas google-genai -q

In [ ]:
# =====================================================================
# DỰ ÁN: FINETUNE NLLB-200 DỊCH SONG NGỮ 2 CHIỀU (Ê ĐÊ <-> VIỆT)
# =====================================================================

import os
import torch
import unicodedata
import pandas as pd
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# ---------------------------------------------------------------------
# 1. KIỂM TRA PHẦN CỨNG GPU T4
# ---------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang chạy trên thiết bị tính toán: {device.upper()}")

# ---------------------------------------------------------------------
# 2. ĐỌC VÀ CHUẨN HÓA DỮ LIỆU
# ---------------------------------------------------------------------
def chuan_hoa_nfc(text):
    if not isinstance(text, str): return ""
    return unicodedata.normalize('NFC', text.strip())

df = pd.read_csv("du_lieu_song_ngu.csv")
df['text_ede'] = df['text_ede'].apply(chuan_hoa_nfc)
df['text_vi']  = df['text_vi'].apply(chuan_hoa_nfc)

# Tạo 2 Dataset riêng biệt cho 2 chiều
ds_ede_vi = Dataset.from_pandas(pd.DataFrame({'src': df['text_ede'], 'tgt': df['text_vi']}))
ds_vi_ede = Dataset.from_pandas(pd.DataFrame({'src': df['text_vi'], 'tgt': df['text_ede']}))

print(f"[*] Đã nạp {len(df)} câu gốc. Đang nhân đôi thành {len(df)*2} câu để huấn luyện 2 chiều...")

# ---------------------------------------------------------------------
# 3. NẠP MÔ HÌNH VÀ TOKENIZER
# ---------------------------------------------------------------------
model_checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

# ---------------------------------------------------------------------
# 4. TOKENIZATION THÔNG MINH CHO 2 CHIỀU
# ---------------------------------------------------------------------
def ma_hoa_ede_vi(examples):
    tokenizer.src_lang = "ind_Latn"
    inputs = tokenizer(examples["src"], max_length=128, truncation=True)
    labels = tokenizer(text_target=examples["tgt"], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

def ma_hoa_vi_ede(examples):
    tokenizer.src_lang = "vie_Latn"
    inputs = tokenizer(examples["src"], max_length=128, truncation=True)
    labels = tokenizer(text_target=examples["tgt"], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

print("[*] Đang mã hóa dữ liệu 2 chiều...")
tokenized_ede_vi = ds_ede_vi.map(ma_hoa_ede_vi, batched=True, remove_columns=["src", "tgt"])
tokenized_vi_ede = ds_vi_ede.map(ma_hoa_vi_ede, batched=True, remove_columns=["src", "tgt"])

# Trộn 2 tập dữ liệu lại và chia Train/Eval
full_tokenized_dataset = concatenate_datasets([tokenized_ede_vi, tokenized_vi_ede]).shuffle(seed=42)
dataset_split = full_tokenized_dataset.train_test_split(test_size=0.2, seed=42)
tokenized_train = dataset_split['train']
tokenized_eval  = dataset_split['test']

# ---------------------------------------------------------------------
# 5 & 6. CẤU HÌNH VÀ BẮT ĐẦU HUẤN LUYỆN
# ---------------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mo_hinh_dich_2_chieu",

    # --- HIỆN TRAINING LOSS LIÊN TỤC ---
    logging_strategy="steps",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=10,

    # --- ÉP TIẾT KIỆM RAM TỐI ĐA (CHỐNG CRASH) ---
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,

    # --- THÔNG SỐ HUẤN LUYỆN CƠ BẢN ---
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=4,             # Giảm xuống 4 epochs tránh overfitting
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

print("\n🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 2 CHIỀU...")
trainer.train()

# Lưu kết quả
final_model_path = "./nllb_2_chieu_hoan_chinh"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\n🎉 HOÀN THÀNH! Mô hình 2 chiều đã lưu tại: {final_model_path}")

[*] Đang chạy trên thiết bị tính toán: CUDA
[*] Đã nạp 408 câu gốc. Đang nhân đôi thành 816 câu để huấn luyện 2 chiều...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

[*] Đang mã hóa dữ liệu 2 chiều...


Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]


🚀 BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH 2 CHIỀU...


Step,Training Loss,Validation Loss
10,46.814508,4.709266
20,38.217593,4.172891
30,33.429977,3.845987
40,32.886606,3.687745
50,30.587344,3.570154
60,28.792108,3.469908
70,30.721704,3.400783
80,27.937109,3.331711
90,25.109312,3.271252
100,26.568933,3.218207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 HOÀN THÀNH! Mô hình 2 chiều đã lưu tại: ./nllb_2_chieu_hoan_chinh


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Trỏ đường dẫn trực tiếp vào Google Drive nơi bạn đã lưu mô hình
model_path = "/content/drive/MyDrive/nllb_ede2vi_hoan_chinh"

# Kiểm tra xem đã kết nối Drive chưa
if not os.path.exists(model_path):
    print(f"[LỖI] Không tìm thấy mô hình tại {model_path}.")
    print("👉 Hãy chạy lệnh này ở 1 ô mới để kết nối lại Drive nhé:")
    print("from google.colab import drive\ndrive.mount('/content/drive')")
else:
    print("[*] Đang nạp bộ não AI từ Google Drive, vui lòng đợi vài giây...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, src_lang="ind_Latn", tgt_lang="vie_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")
    print("[*] Tải mô hình thành công!\n")

    def dich_tieng_ede(cau_ede):
        inputs = tokenizer(cau_ede.strip(), return_tensors="pt").to(model.device)
        ma_tieng_viet = tokenizer.convert_tokens_to_ids("vie_Latn")

        with torch.no_grad():
            outputs = model.generate(**inputs, forced_bos_token_id=ma_tieng_viet, max_length=128)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Giao diện nhập câu thử nghiệm
    cau_nhap = input("👉 Nhập câu tiếng Ê Đê: ")
    if cau_nhap.strip():
        ket_qua = dich_tieng_ede(cau_nhap)
        print(f"🤖 AI dịch sang Tiếng Việt: {ket_qua}")

[LỖI] Không tìm thấy mô hình tại /content/drive/MyDrive/nllb_ede2vi_hoan_chinh.
👉 Hãy chạy lệnh này ở 1 ô mới để kết nối lại Drive nhé:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import gradio as gr
import torch
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

# 1. Kết nối Google Drive với Colab
drive.mount('/content/drive')

# 2. Khai báo đường dẫn tới thư mục mô hình trên Drive
model_path = "/content/drive/MyDrive/nllb_ede2vi_hoan_chinh"

# Kiểm tra thư mục có tồn tại hay không
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Không tìm thấy thư mục mô hình tại: {model_path}. Vui lòng kiểm tra lại!")

# 3. Nạp mô hình và tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

# 4. Hàm dịch 2 chiều
def dich_2_chieu(van_ban, chieu_dich):
    if not van_ban.strip():
        return "Vui lòng nhập văn bản..."

    # Xác định mã ngôn ngữ
    if chieu_dich == "Ê Đê ➔ Tiếng Việt":
        src = "ind_Latn"
        tgt = "vie_Latn"
    else: # Tiếng Việt ➔ Ê Đê
        src = "vie_Latn"
        tgt = "ind_Latn"

    # Cài đặt ngôn ngữ nguồn
    tokenizer.src_lang = src
    inputs = tokenizer(van_ban.strip(), return_tensors="pt").to(model.device)
    ma_ngon_ngu_dich = tokenizer.convert_tokens_to_ids(tgt)

    with torch.no_grad():
        outputs = model.generate(**inputs, forced_bos_token_id=ma_ngon_ngu_dich, max_length=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 5. Xây dựng giao diện Web
giao_dien = gr.Interface(
    fn=dich_2_chieu,
    inputs=[
        gr.Textbox(lines=3, placeholder="Nhập câu cần dịch vào đây...", label="Văn bản gốc"),
        gr.Radio(["Ê Đê ➔ Tiếng Việt", "Tiếng Việt ➔ Ê Đê"], value="Ê Đê ➔ Tiếng Việt", label="Chọn chiều dịch")
    ],
    outputs=gr.Textbox(lines=3, label="Kết quả dịch (AI)"),
    title="🌟 Hệ thống Dịch tự động Song ngữ Ê Đê ⬌ Việt",
    description="Dự án KHKT - Mô hình AI NLLB-200 Fine-tuned hỗ trợ dịch 2 chiều."
)

giao_dien.launch(share=True)

Mounted at /content/drive


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: (temporary share link removed)

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Cài đặt thư viện tạo giao diện
!pip install gradio -q

In [ ]:
# Cài đặt thư viện đánh giá
!pip install evaluate sacrebleu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.5 MB/s eta 0:00:00


In [ ]:
import evaluate
import pandas as pd
from tqdm import tqdm
from google.colab import files
import os
import torch

# 1. Kiểm tra và yêu cầu tải lại file CSV nếu bị Colab xóa mất
if not os.path.exists("du_lieu_song_ngu.csv"):
    print("[!] Không tìm thấy file dữ liệu. Bạn hãy bấm nút bên dưới để tải lại file 'du_lieu_song_ngu.csv' lên nhé:")
    uploaded = files.upload()
    ten_file = list(uploaded.keys())[0]
    if ten_file != "du_lieu_song_ngu.csv":
        os.rename(ten_file, "du_lieu_song_ngu.csv")

# 2. Tải công cụ tính điểm BLEU
bleu_metric = evaluate.load("sacrebleu")

# 3. Đọc file dữ liệu CSV để lấy câu test
df = pd.read_csv("du_lieu_song_ngu.csv")

# Lấy 15 câu đầu tiên làm bài kiểm tra cho AI
tap_kiem_tra = df.head(15)

cau_ai_dich = []
cau_con_nguoi_dich = []

# --- BỔ SUNG HÀM DỊCH ĐỂ SỬA LỖI NAME ERROR ---
def dich_ede_sang_viet(cau_ede):
    tokenizer.src_lang = "ind_Latn"
    inputs = tokenizer(cau_ede.strip(), return_tensors="pt").to(model.device)
    ma_ngon_ngu_dich = tokenizer.convert_tokens_to_ids("vie_Latn")

    with torch.no_grad():
        outputs = model.generate(**inputs, forced_bos_token_id=ma_ngon_ngu_dich, max_length=128)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# ----------------------------------------------

print("\n[*] Bắt đầu cho AI làm bài thi dịch...")
for index, row in tqdm(tap_kiem_tra.iterrows(), total=len(tap_kiem_tra)):
    cau_ede = str(row['text_ede'])
    cau_vi_chuan = str(row['text_vi'])

    # AI tự dịch bằng hàm đã tạo ở trên
    cau_dich = dich_ede_sang_viet(cau_ede)

    cau_ai_dich.append(cau_dich)
    cau_con_nguoi_dich.append([cau_vi_chuan])

# 4. Tính điểm
results = bleu_metric.compute(predictions=cau_ai_dich, references=cau_con_nguoi_dich)

print("\n" + "="*50)
print(f"📊 ĐIỂM BLEU CỦA MÔ HÌNH: {results['score']:.2f} / 100")
print("="*50)
print("\n[Ghi chú cho báo cáo KHKT]:")
print("- Điểm BLEU (từ 0-100) đo mức độ trùng khớp giữa câu AI dịch và câu chuẩn do con người dịch.")
print("- Điểm số này là minh chứng định lượng cho thấy mô hình AI hoạt động hiệu quả.")


[*] Bắt đầu cho AI làm bài thi dịch...


100%|██████████| 15/15 [00:18<00:00,  1.21s/it]


📊 ĐIỂM BLEU CỦA MÔ HÌNH: 12.46 / 100

[Ghi chú cho báo cáo KHKT]:
- Điểm BLEU (từ 0-100) đo mức độ trùng khớp giữa câu AI dịch và câu chuẩn do con người dịch.
- Điểm số này là minh chứng định lượng cho thấy mô hình AI hoạt động hiệu quả.
